# NRCH two-loop flow — Julia + Makie

Recreates the 3D flow plot and the `bu = 0` isosurface from `NRCH_twoloop.ipynb`.
Loads the integrals precomputed by the Python notebook from `data/samples.npz`.

In [ ]:
# Run once to install dependencies.
using Pkg
# Pkg.add(["NPZ", "Interpolations"|, "Meshing", "GeometryBasics", "LaTeXStrings"])

In [ ]:

using NPZ
using Interpolations
using Meshing
using GeometryBasics
using Random
using LaTeXStrings

using GLMakie
GLMakie.activate!()
Makie.inline!(false)
show_fig(fig) = display(fig)

In [ ]:
data = npzread("data/samples.npz")

b_samples = data["b_vals"]
I_ff   = data["I_ff_real"]   .+ im .* data["I_ff_imag"]
I_dff  = data["I_dff_real"]  .+ im .* data["I_dff_imag"]
I_ddff = data["I_ddff_real"] .+ im .* data["I_ddff_imag"]

length(b_samples), extrema(b_samples)

In [ ]:
B(b) = (1 - im*b) / (1 + im*b)

g_samples = @. (1 + 8 * (1 + B(b_samples)) * (I_dff + 2 * B(b_samples) * I_ddff)) / (1 + im * b_samples)

# Cubic spline interpolation of the real and imaginary parts on the uniform grid.
g_re_itp = cubic_spline_interpolation(range(b_samples[1], b_samples[end]; length=length(b_samples)), real.(g_samples))
g_im_itp = cubic_spline_interpolation(range(b_samples[1], b_samples[end]; length=length(b_samples)), imag.(g_samples))

gs(b) = g_re_itp(b) + im * g_im_itp(b)

gs(0.0)  # should be ~1/3

In [ ]:
d = 3
ϵ = 4 - d

# a = λ, b = κ, c = μ
h(a, b) = (1 + b*a) * real(gs(b)) + (b - a) * imag(gs(b))

βu(u, a, b, c) = +ϵ*u - 10*u^2 * (1 - 1/5 * (a-b)^2 / (1+b^2))
βa(u, a, b, c) = -2*u*(a - b) * (1+a^2) / (1+b^2)
βb(u, a, b, c) = -2*u^2 * (b - a) * (1 + h(a, b))
βc(u, a, b, c) = +4 * u * (c - a)

# Flow on the RG "upward" direction (negative beta).
v(u, a, b, c) = (βu(u,a,b,c), βa(u,a,b,c), βb(u,a,b,c), βc(u,a,b,c))

eq(u, a, b) = βu(u, a, b, 0.0)

In [ ]:
function simulate(v, u0, a0, b0, c0; dt=0.02, nsteps=1000)
    traj = zeros(nsteps, 4)
    traj[1, :] = [u0, a0, b0, c0]
    for i in 2:nsteps
        u, a, b, c = traj[i-1, :]
        du, da, db, dc = v(u, a, b, c)
        traj[i, :] = [u + dt*du, a + dt*da, b + dt*db, c + dt*dc]
    end
    return traj
end

function get_init_rnd(N, ur, ar, br, cr; rng=Random.default_rng(12))
    inits = Vector{NTuple{4,Float64}}()
    for _ in 1:N
        u0 = rand(rng) * ur
        a0 = (2*rand(rng) - 1) * ar
        b0 = (2*rand(rng) - 1) * br 
        c0 = rand(rng)*cr
        push!(inits, (u0, a0, b0, c0))
    end
    return inits
end

In [ ]:
# Per-axis data lengths that come out equal on screen. Axis3 scales the data by
# 2/width per axis for aspect = :equal, by 2/max(width) for :data, and by
# 2/width * a/max(a) for an explicit aspect tuple a — this is the inverse of
# that (up to the common factor 2). Dividing by it lands you in the space where
# the axis box is drawn as a cube, which is where the cone has to be built for
# its base to come out round on screen, whatever the aspect setting is.
function box_units(lims, asp)
    ws = widths(lims)
    if asp === :equal
        return Vec3f(ws[1], ws[2], ws[3])
    elseif asp === :data
        m = maximum(ws)
        return Vec3f(m, m, m)
    elseif asp isa Tuple || asp isa VecTypes{3}
        return Vec3f((ws .* maximum(asp) ./ asp)...)
    else
        error("unsupported Axis3 aspect: $asp")
    end
end

# One cone head, pointing from p_prev to p_curr: a ring of triangles fanning out
# to the tip. Built in box units (divide by `w`) and scaled back at the end, so
# the base is round on screen whatever the axis ranges are.
#
# `scale` sizes the head as a fraction of the on-screen box — the numbers below
# are the old data-unit values divided by 2.4, the box width of the abc figures,
# so those come out as before.
#
# A zero-length direction gives a collapsed (invisible) mesh rather than an
# error, which is what the growing-trajectory animation needs for a trajectory
# that has not started yet.
function cone_mesh(p_prev, p_curr, w; scale=1, n_sides=10)
    head_len    =  0.075  * scale
    head_radius =  0.0333 * scale
    head_shift  = -0.0417 * scale

    _len(p)      = sqrt(p[1]^2 + p[2]^2 + p[3]^2)
    _cross(a, b) = Vec3f(a[2]*b[3] - a[3]*b[2], a[3]*b[1] - a[1]*b[3], a[1]*b[2] - a[2]*b[1])

    faces = [GLTriangleFace(1, i + 1, (i % n_sides) + 2) for i in 1:n_sides]

    dir = (p_curr .- p_prev) ./ w
    _len(dir) > 1e-12 || return GeometryBasics.Mesh(fill(Point3f(p_curr), n_sides + 1), faces)
    u = dir ./ _len(dir)

    # Two directions orthogonal to u, to sweep the ring with.
    ref = abs(u[3]) > 0.9 ? Vec3f(1, 0, 0) : Vec3f(0, 0, 1)
    n = _cross(u, ref); n = n ./ _len(n)
    b = _cross(u, n)

    tip  = p_curr ./ w .- head_shift .* u
    base = tip .- head_len .* u
    ring = [Point3f((base .+ head_radius .* (cos(θ) .* n .+ sin(θ) .* b)) .* w)
            for θ in range(0, 2π; length=n_sides + 1)[1:end-1]]

    return GeometryBasics.Mesh(vcat([Point3f(tip .* w)], ring), faces)
end

# Cone head at the end of each segment, pointing along its last step. The
# geometry is rebuilt whenever the limits or the aspect change, so it does not
# matter that this is called from inside the plot_flow* functions, i.e. before
# xlims!/ylims!/zlims! have run.
function add_arrows(segs, ax; arrow_col=nothing, indx=(2,3,4), scale=1)

    head_colors = arrow_col === nothing ? fill(:black, length(segs)) : collect(arrow_col)

    for (idx, seg) in enumerate(segs)
        size(seg, 1) >= 2 || continue
        p_prev = Vec3f(seg[end-1, indx[1]], seg[end-1, indx[2]], seg[end-1, indx[3]])
        p_curr = Vec3f(seg[end,   indx[1]], seg[end,   indx[2]], seg[end,   indx[3]])

        head = lift(ax.finallimits, ax.aspect) do lims, asp
            cone_mesh(p_prev, p_curr, box_units(lims, asp); scale=scale)
        end

        mesh!(ax, head; color=head_colors[idx], shading=NoShading, fxaa=true)
    end
end

In [ ]:
function plot_flow3D!(ax, v, N, trajs;
                       nsteps=1000, cmap=:plasma, crange=(-10.0, 10.0),
                       seed=nothing, linewidth=3, subsample=5, ms=10, 
                       arrow=false, arrowscale=1, endpoint=true)

    cmin, cmax = Float32(crange[1]), Float32(crange[2])

    # Build one big NaN-separated strip so we get a single lines! draw call.
    # NOTE: NaN goes only in x/y/z (that's what breaks the line); the color array
    # uses a finite sentinel because NaN in per-vertex color can confuse shaders.
    segs = [traj[1:subsample:end, :] for traj in trajs]
    npts = sum(size(s, 1) for s in segs) + length(segs)  # +1 sep per traj

    xs = Vector{Float32}(undef, npts)
    ys = Vector{Float32}(undef, npts)
    zs = Vector{Float32}(undef, npts)
    cs = Vector{Float32}(undef, npts)

    i = 1
    for s in segs
        n = size(s, 1)
        @inbounds for k in 1:n
            xs[i] = s[k, 2]
            ys[i] = s[k, 3]
            zs[i] = s[k, 1]
            cs[i] = clamp(Float32(s[k, 4]), cmin, cmax)
            i += 1
        end
        # Strip separator: NaN in position only.
        xs[i] = NaN32; ys[i] = NaN32; zs[i] = NaN32; cs[i] = cmin
        i += 1
    end

    start_x = Float32[t[1, 2]     for t in trajs]
    start_y = Float32[t[1, 3]     for t in trajs]
    start_z = Float32[t[1, 1]     for t in trajs]
    end_x   = Float32[t[end, 2]   for t in trajs]
    end_y   = Float32[t[end, 3]   for t in trajs]
    end_z   = Float32[t[end, 1]   for t in trajs]
    end_c   = Float32[clamp(Float32(t[end, 4]), cmin, cmax) for t in trajs]

    lines!(ax, xs, ys, zs; 
            color=cs, colormap=cmap, colorrange=crange, linewidth=linewidth, fxaa=true)
    scatter!(ax, start_x, start_y, start_z; color=:blue, markersize=ms)
    
    if endpoint;
        scatter!(ax, end_x, end_y, end_z;
                color=end_c, colormap=cmap, colorrange=crange, markersize=ms)
    end
    if arrow; add_arrows(segs,ax,indx=(2,3,1),scale=arrowscale); end
end

In [ ]:
function plot_flowabc!(ax, v, N, trajs;
                       nsteps=1000, cmap=:plasma, crange=(-10.0, 10.0),
                       linewidth=3, subsample=5, ms=10, arrow_col=nothing, 
                       arrowscale=1)

    cmin, cmax = Float32(crange[1]), Float32(crange[2])

    segs = [traj[1:subsample:end, :] for traj in trajs]
    npts = sum(size(s, 1) for s in segs) + length(segs)  # +1 sep per traj

    xs = Vector{Float32}(undef, npts)
    ys = Vector{Float32}(undef, npts)
    zs = Vector{Float32}(undef, npts)
    cs = Vector{Float32}(undef, npts)

    i = 1
    for s in segs
        n = size(s, 1)
        @inbounds for k in 1:n
            xs[i] = s[k, 2]
            ys[i] = s[k, 3]
            zs[i] = s[k, 4]
            i += 1
        end
        # Strip separator: NaN in position only.
        xs[i] = NaN32; ys[i] = NaN32; zs[i] = NaN32; cs[i] = cmin
        i += 1
    end

    start_x = Float32[t[1, 2]     for t in trajs]
    start_y = Float32[t[1, 3]     for t in trajs]
    start_z = Float32[t[1, 4]     for t in trajs]
    end_x   = Float32[t[end, 2]   for t in trajs]
    end_y   = Float32[t[end, 3]   for t in trajs]
    end_z   = Float32[t[end, 4]   for t in trajs]

    lines!(ax, xs, ys, zs; 
            color=zs, colormap=cmap, colorrange=crange, linewidth=linewidth, fxaa=true)
    scatter!(ax, start_x, start_y, start_z;color=:blue, markersize=ms)
    add_arrows(segs,ax,arrow_col=arrow_col, scale=arrowscale)
end

In [ ]:
yellow      = colorant"#fff800"
pink        = colorant"#f03ed4"
blue        = colorant"#a3f8ff"
green       = colorant"#c8ffce"
turquise    = colorant"#41e3c0"
red         = RGBf(0.8,0.2,0.2)

mapBlack = cgrad([:black, :black])
cmap = cgrad([blue, blue, pink, green, green], [0.0, 0.1, .5, .9, 1.0])

In [ ]:
set_theme!(theme_latexfonts())

l = 1.85
crange = (-l, l)

ur = .2
ar = l
br = 1.1
cr = ar

s = 15
fs = 50
ls = 50
ms = 30
lw = 12

u_grid = collect(range(0.01, ur;    length=30))
a_grid = collect(range(-0.8*ar, ar; length=60))
b_grid = collect(range(-0.1*br, br; length=60))

V = [eq(u, a, b) for a in a_grid, b in b_grid, u in u_grid]


function get_init(N, ur, ar, br, cr; rng=Random.default_rng())
    return [
        (0.01,  +0.5,     0,    +0.33,),
        (0.01,  -0.5,     0,    -0.33,),
        (0.20,  -0.5,     1.,   -0.1782, ),
        (0.20,  +1 ,    0.,    +0., ),
        # (0.05,  -1 ,    0.5,    -0.471, ),
        (0.05,  -1 ,    0.5,    -0.46745, ),
    ]
end

N = get_init(0,0,0,0,0);
nsteps = 800

rng = Random.MersenneTwister(1)
inits = get_init(N, ur, ar, br, cr; rng=rng)
trajs = [simulate(v, u0, a0, b0, c0; nsteps=nsteps) for (u0, a0, b0, c0) in inits];

In [ ]:
# a b c

fig2 = Figure(size=(150*s, 100*s); fontsize = 30)
ax2  = Axis3(fig2[1, 1]; 
    xlabel=L"\lambda", ylabel=L"\kappa", zlabel=L"\mu",
    viewmode = :fit,aspect = :equal,
    xlabelsize = fs, ylabelsize = fs, zlabelsize = fs, 
    xlabeloffset = 50, ylabeloffset = 50, zlabeloffset = 70,
    )

# Make the scaling equal on all axes.
xlims!(ax2, (-ar, ar))
ylims!(ax2, (-ar, ar))
zlims!(ax2, (-ar, ar))
ax2.aspect = :data

vals = range(-ar, ar; length=200)
lines!(ax2, vals, vals, vals; color=red, linewidth=10)
 
# Plot the flow trajectories.
plot_flowabc!(ax2, v, N, trajs; linewidth=lw, ms=ms, cmap=cmap, crange=crange)

show_fig(fig2)

In [ ]:
save("plot1.png", fig2; px_per_unit = 4)

In [ ]:
# λ a b

fig1 = Figure(size=(150*s, 100*s); fontsize = 30)
ax1  = Axis3(fig1[1, 1]; 
    # xlabel=L"\lambda", ylabel=L"\kappa", zlabel=L"\upsilon", 
    xlabel=L"\alpha_1/u", ylabel=L"\beta_0/K", zlabel=L"u", 
    viewmode = :fit, aspect=:equal,
    xlabelsize = fs, ylabelsize = fs, zlabelsize = fs,
    xlabeloffset = 50, ylabeloffset = 50, zlabeloffset = 70,
    )

plot_flow3D!(
    ax1, v, N, trajs;
    nsteps=nsteps, cmap=cmap, seed=nothing, linewidth=lw, subsample=5, ms=ms, crange=crange
    )

pts, tris = isosurface(V, MarchingCubes(iso=0.0), a_grid, b_grid, u_grid)
verts = [Point3f(p...) for p in pts]
faces = [TriangleFace{Int}(f...) for f in tris]
surface_mesh = GeometryBasics.Mesh(verts, faces)

color = :steelblue3

mesh!(ax1, surface_mesh; color=(color, .4), transparency=true)
Colorbar(fig1[1, 2]; 
        colormap=cmap, limits=crange, label=L"\mu", width=50, height = 400, 
        tellheight = false, valign = :center, ticklabelsize=fs, labelsize=fs)

vals = range(-.11, br; length=200)
lines!(ax1, vals, vals, vals ./ vals .* 0.1; color=red, linewidth=10)

zlims!(ax1, (0, 1.1*ur))

show_fig(fig1)

In [ ]:
save("plot2.png", fig1; px_per_unit = 4)

# Plot flow in abc with arrows

In [ ]:
ur = .2
ar = 1.2
br = ar
cr = ar

function get_init(N, ur, ar, br, cr; rng=Random.default_rng())
    return [
        (0.1,  +1,     0,    +.6,),
        (0.1,  -1,     0,    -.6,),
        (0.1,  -1,     0,    +.6,),
        (0.1,  +1,     0,    -.6,),
        (0.1,  1,     -1,    .475,),
    ]
end
N = get_init(0,0,0,0,0)


fig = Figure(size=(150*s, 100*s); fontsize = 30)
ax  = Axis3(fig[1, 1]; viewmode = :fit,
    xlabel=L"\lambda=\alpha_1/u", ylabel=L"\kappa=\beta_0/K", zlabel=L"\mu=\alpha_0/r",
    xlabelsize = fs, ylabelsize = fs, zlabelsize = fs,
    xlabeloffset = 50, ylabeloffset = 50, zlabeloffset = 70,
    )

xlims!(ax, (-ar, ar))
ylims!(ax, (-ar, ar))
zlims!(ax, (-ar, ar))

ax.aspect = :data

vals = range(-ar, ar; length=200)
lines!(ax, vals, vals, vals; color=red, linewidth=12)

nsteps = 500
inits = get_init(N, ur, ar, br, cr; rng=rng)
trajs = [simulate(v, u0, a0, b0, c0; nsteps=nsteps) for (u0, a0, b0, c0) in inits]

plot_flowabc!(ax, v, N, trajs ; 
    nsteps=500, linewidth=lw, ms=ms, arrow_col=(blue, green, green, blue, pink),cmap=mapBlack)

show_fig(fig)

In [ ]:
save("plot.png", fig; px_per_unit = 4)

# Random sample

In [ ]:
# λ a b

l = 2
crange = (-l, l)

ur = .4
ar = 1.6
br = 1.6
cr = 0.

N = 10
fs = 50
ms = 20
lw = 7

nsteps = 400

rng = Random.MersenneTwister()
inits = get_init_rnd(N, ur, ar, br, cr; rng=rng)
trajs = [simulate(v, u0, a0, b0, c0; nsteps=nsteps) for (u0, a0, b0, c0) in inits];

fig1 = Figure(size=(150*s, 100*s); fontsize = 30)
ax1  = Axis3(fig1[1, 1]; 
    xlabel=L"a", ylabel=L"b", zlabel=L"\lambda", 
    viewmode = :fit, aspect=:equal,
    xlabelsize = fs, ylabelsize = fs, zlabelsize = fs,
    xlabeloffset = 50, ylabeloffset = 50, zlabeloffset = 70,
    )

# No box: drop the background panels, grid, ticks, spines and axis labels.
hidedecorations!(ax1)
ax1.xspinesvisible = false
ax1.yspinesvisible = false
ax1.zspinesvisible = false
ax1.xypanelvisible = false
ax1.xzpanelvisible = false
ax1.yzpanelvisible = false

plot_flow3D!(
    ax1, v, N, trajs;
    nsteps=nsteps, cmap=cmap, seed=nothing, linewidth=lw, subsample=5, ms=ms, crange=crange, arrow=true, arrowscale=0.4, endpoint=false
)

u_grid = collect(range(0.01, ur;    length=30))
a_grid = collect(range(-1.0*ar, 1.0*ar; length=60))
b_grid = collect(range(-1.0*br, 1.0*br; length=60))

V = [eq(u, a, b) for a in a_grid, b in b_grid, u in u_grid]

pts, tris = isosurface(V, MarchingCubes(iso=0.0), a_grid, b_grid, u_grid)
verts = [Point3f(p...) for p in pts]
faces = [TriangleFace{Int}(f...) for f in tris]
surface_mesh = GeometryBasics.Mesh(verts, faces)

color = :steelblue3

mesh!(ax1, surface_mesh; color=(color, .4), transparency=true)

zlims!(ax1, (0, 1.1*ur))

show_fig(fig1)

In [ ]:
save("diss.png", fig1; px_per_unit = 4)

In [ ]:
# λ a b

l = 2
crange = (-l, l)

ur = .25
ar = 1.8
br = 1.8
cr = 0.

N = 20
fs = 50
ms = 20
lw = 4

nsteps = 500

rng = Random.MersenneTwister()
inits = get_init_rnd(N, ur, ar, br, cr; rng=rng)
trajs = [simulate(v, u0, a0, b0, c0; nsteps=nsteps) for (u0, a0, b0, c0) in inits];

fig1 = Figure(size=(150*s, 100*s); fontsize = 30)
ax1  = Axis3(fig1[1, 1]; 
    xlabel=L"a", ylabel=L"b", zlabel=L"\lambda", 
    viewmode = :fit, aspect=:equal,
    xlabelsize = fs, ylabelsize = fs, zlabelsize = fs,
    xlabeloffset = 50, ylabeloffset = 50, zlabeloffset = 70,
    )

# No box: drop the background panels, grid, ticks, spines and axis labels.
# hidedecorations!(ax1)
# ax1.xspinesvisible = false
# ax1.yspinesvisible = false
# ax1.zspinesvisible = false

plot_flow3D!(
    ax1, v, N, trajs;
    nsteps=nsteps, cmap=cmap, seed=nothing, linewidth=lw, subsample=5, ms=ms, crange=crange, arrow=true, arrowscale=0.4, endpoint=false
)

zlims!(ax1, (0, 1.1*ur))

show_fig(fig1)

# Video

In [ ]:
# Animation machinery: trajectories growing out of their initial conditions.
# Nothing here is specific to a figure — plot_growing_flow! takes the same
# arguments as plot_flow3D!, so any of the figures above can be animated by
# swapping the one call and then recording.

# Same picture as plot_flow3D! (blue start dots, one NaN-separated line strip,
# optional end dot and cone head), but cut off at a progress value in 0..1.
# Returns that Observable: 0 shows only the initial conditions, 1 the finished
# trajectories, anything between is a frame of the animation.
#
# The strip keeps its full length throughout and the not-yet-drawn tail is
# masked with NaN *positions*; the colour vector therefore never changes length,
# so Makie never sees a positions/colours mismatch mid-update.
#
# indx picks the (x, y, z) columns of a trajectory row and cidx the column the
# colormap reads — (2, 3, 1) and 4 match plot_flow3D!.
function plot_growing_flow!(ax, trajs;
                            indx=(2, 3, 1), cidx=4,
                            cmap=:plasma, crange=(-10.0, 10.0),
                            linewidth=3, subsample=5, ms=10,
                            endpoint=true, arrow=false, arrowscale=1,
                            arrow_col=nothing, start_col=:blue)

    cmin, cmax = Float32(crange[1]), Float32(crange[2])
    nan3 = Point3f(NaN32, NaN32, NaN32)

    segs = [traj[1:subsample:end, :] for traj in trajs]
    ns   = [size(s, 1) for s in segs]
    npts = sum(ns) + length(segs)          # +1 separator per trajectory

    pts  = Vector{Point3f}(undef, npts)    # the finished strip, never mutated
    cs   = Vector{Float32}(undef, npts)
    offs = Vector{Int}(undef, length(segs))  # index of each trajectory's first point

    i = 1
    for (k, s) in enumerate(segs)
        offs[k] = i
        @inbounds for j in 1:ns[k]
            pts[i] = Point3f(s[j, indx[1]], s[j, indx[2]], s[j, indx[3]])
            cs[i]  = clamp(Float32(s[j, cidx]), cmin, cmax)
            i += 1
        end
        pts[i] = nan3; cs[i] = cmin        # strip separator
        i += 1
    end

    progress = Observable(0.0)
    nvis(p, k) = clamp(floor(Int, p * ns[k]), 0, ns[k])   # samples shown of trajectory k

    visible = lift(progress) do p
        out = copy(pts)
        for k in eachindex(segs), j in (nvis(p, k) + 1):ns[k]
            out[offs[k] + j - 1] = nan3
        end
        out
    end
    lines!(ax, visible; color=cs, colormap=cmap, colorrange=crange,
           linewidth=linewidth, fxaa=true)

    scatter!(ax, [pts[offs[k]] for k in eachindex(segs)];
             color=start_col, markersize=ms)

    if endpoint
        # Dot riding the growing tip; NaN keeps it hidden until the line starts.
        tip_pos = lift(progress) do p
            [nvis(p, k) < 1 ? nan3 : pts[offs[k] + nvis(p, k) - 1] for k in eachindex(segs)]
        end
        tip_col = lift(progress) do p
            [cs[offs[k] + max(nvis(p, k), 1) - 1] for k in eachindex(segs)]
        end
        scatter!(ax, tip_pos; color=tip_col, colormap=cmap, colorrange=crange, markersize=ms)
    end

    if arrow
        head_colors = arrow_col === nothing ? fill(:black, length(segs)) : collect(arrow_col)
        for k in eachindex(segs)
            head = lift(ax.finallimits, ax.aspect, progress) do lims, asp, p
                w = box_units(lims, asp)
                n = nvis(p, k)
                # Fewer than two samples: collapse the cone onto the start point,
                # which draws nothing.
                a, b = n < 2 ? (pts[offs[k]], pts[offs[k]]) :
                               (pts[offs[k] + n - 2], pts[offs[k] + n - 1])
                cone_mesh(a, b, w; scale=arrowscale)
            end
            mesh!(ax, head; color=head_colors[k], shading=NoShading, fxaa=true)
        end
    end

    return progress
end

# The bu = 0 surface as a mesh, on a grid sized to the given ranges — the same
# construction the figures above do inline, pulled out so a figure cell is one
# call. n_u / n_ab are the marching-cubes grid resolutions.
function isosurface_mesh(ur, ar, br; n_u=30, n_ab=60, iso=0.0)
    u_grid = collect(range(0.01, ur; length=n_u))
    a_grid = collect(range(-ar, ar;  length=n_ab))
    b_grid = collect(range(-br, br;  length=n_ab))

    V = [eq(u, a, b) for a in a_grid, b in b_grid, u in u_grid]
    pts, tris = isosurface(V, MarchingCubes(iso=iso), a_grid, b_grid, u_grid)

    return GeometryBasics.Mesh([Point3f(p...) for p in pts],
                               [TriangleFace{Int}(f...) for f in tris])
end

# Pin the axis to the limits it has right now. Needed before recording: the
# growing strip masks its tail with NaN, so under autolimits the box would
# shrink to whatever is currently visible and jitter through the whole video.
# Call it with the animation wound to the end (progress[] = 1).
function freeze_limits!(ax)
    Makie.reset_limits!(ax)   # autolimits are otherwise only computed at display time
    r = ax.finallimits[]
    lo, hi = minimum(r), maximum(r)
    xlims!(ax, (lo[1], hi[1]))
    ylims!(ax, (lo[2], hi[2]))
    zlims!(ax, (lo[3], hi[3]))
    return ax
end

# Bounding box of the ink in a rendered frame, as fractions of the frame: the
# smallest box containing every pixel that is not the white background. Given
# `ax` and `azimuths`, the union of those boxes over the camera angles — the
# part of the frame a spinning video ever draws in.
#
# The box is measured off a rendered frame rather than computed from the layout
# because the white border comes from three places at once: figure_padding, the
# protrusions Axis3 reserves for its decorations, and above all viewmode = :fit,
# which scales the box so that it fits at *every* rotation and therefore always
# keeps slack around it. That slack is a fixed fraction of the frame, so no
# choice of figure size removes it — the frame has to be cut instead.
function ink_box(fig, ax = nothing, azimuths = nothing; white = 0.99, pad = 0.002)
    x0, y0, x1, y1 = Inf, Inf, -Inf, -Inf
    az0 = ax === nothing ? nothing : ax.azimuth[]
    isink(c) = c.r < white || c.g < white || c.b < white

    for a in (ax === nothing || azimuths === nothing ? (nothing,) : azimuths)
        a === nothing || (ax.azimuth[] = a)
        img = RGBf.(colorbuffer(fig))          # rows = y from the top, cols = x
        rows = [i for i in axes(img, 1) if any(j -> isink(img[i, j]), axes(img, 2))]
        isempty(rows) && continue
        cols = [j for j in axes(img, 2) if any(i -> isink(img[i, j]), axes(img, 1))]
        h, w = size(img)
        x0 = min(x0, (first(cols) - 1) / w); x1 = max(x1, last(cols) / w)
        y0 = min(y0, (first(rows) - 1) / h); y1 = max(y1, last(rows) / h)
    end

    az0 === nothing || (ax.azimuth[] = az0)
    isfinite(x0) || return nothing
    return (max(x0 - pad, 0.0), max(y0 - pad, 0.0), min(x1 + pad, 1.0), min(y1 + pad, 1.0))
end

# Re-encode `path` cropped to `box` (fractions of the frame, as ink_box returns
# them). The crop is written as an ffmpeg expression in in_w/in_h, so the frame
# size never has to be known here — which matters because a HiDPI screen records
# at px_per_unit > 1, i.e. at more pixels than the figure size. Every number is
# rounded down to an even one, which h264 requires. The uncropped file is kept
# until ffmpeg succeeds, so a failure cannot lose the recording.
function crop_video(path, box; compression = 20)
    x0, y0, x1, y1 = box
    even(e) = "trunc($e/2)*2"
    vf = "crop=" * even("in_w*$(x1 - x0)") * ":" * even("in_h*$(y1 - y0)") * ":" *
                   even("in_w*$x0")        * ":" * even("in_h*$y0")

    uncropped = joinpath(dirname(abspath(path)), "." * basename(path) * ".uncropped")
    mv(path, uncropped; force = true)
    try
        run(pipeline(`$(Makie.FFMPEG_jll.ffmpeg()) -y -loglevel error -i $uncropped
                      -vf $vf -c:v libx264 -crf $compression -pix_fmt yuv420p -preset slow
                      $path`))
    catch err
        mv(uncropped, path; force = true)
        rethrow(err)
    end
    rm(uncropped; force = true)
    return path
end

# Sweep `progress` from 0 to 1 into a video file (.mp4, .gif, .webm — whatever
# FFMPEG understands). hold_start / hold_end are extra still frames at the ends,
# so the video opens on the initial conditions and closes on the finished flow.
#
# Pass `ax` with a `turn_speed` to spin the camera around the z axis while
# recording. turn_speed is in revolutions per second of video, so 0.05 is one
# turn every 20 s — that is the knob to tune. The spin runs through the hold
# frames as well, and exactly one turn over the whole clip is
# turn_speed = framerate / (hold_start + nframes + hold_end).
# `compression` is passed straight to Makie's record (h264: 0 = lossless, 51 =
# worst); raise it if the file comes out large — the isosurface in particular
# encodes badly.
#
# `crop` trims the white border off the finished video. The box is measured
# before recording, at progress = 1 (the picture only ever grows, so that frame
# contains all the ink) and over `crop_samples` of the azimuths the spin
# actually visits, so nothing is cut at any point of the clip; `crop_pad` is the
# margin left around it, in fractions of the frame. This costs one extra encode
# pass — pass crop=false to keep the frame as recorded.
function record_growth(fig, progress, path;
                       nframes=180, framerate=30, hold_start=15, hold_end=30,
                       ax=nothing, turn_speed=0.0, compression=20,
                       crop=true, crop_samples=24, crop_pad=0.002)
    total = hold_start + nframes + hold_end
    az0   = ax === nothing ? 0.0 : ax.azimuth[]
    azimuth_at(i) = az0 + 2π * turn_speed * (i - 1) / framerate

    box = nothing
    if crop
        p0 = progress[]
        progress[] = 1.0
        frames = unique(round.(Int, range(1, total; length = min(total, crop_samples))))
        box = ink_box(fig, ax, azimuth_at.(frames); pad = crop_pad)
        progress[] = p0
    end

    record(fig, path, 1:total; framerate=framerate, compression=compression) do i
        progress[] = clamp((i - hold_start) / nframes, 0.0, 1.0)
        ax === nothing || (ax.azimuth[] = azimuth_at(i))
    end

    box === nothing || crop_video(path, box; compression = compression)
    return path
end

In [ ]:
trajs_vid = [simulate(v, u0, a0, b0, c0; nsteps=nsteps) for (u0, a0, b0, c0) in inits]

# video scale: frame is 100*vs x 100*vs px before record_growth crops it
vs = s

In [ ]:
# Video of the λ a b figure above
# figure_padding = 0 and a square frame keep the white border small to begin
# with — the 3:2 frame was mostly empty on the left and right. record_growth
# cuts off whatever is left once the video is recorded.
fig_vid = Figure(size=(100*vs, 100*vs); fontsize = 30, figure_padding = 0)
ax_vid  = Axis3(fig_vid[1, 1];
    xlabel=L"\lambda", ylabel=L"\kappa", zlabel=L"\upsilon",
    viewmode = :fit, aspect=:equal,
    xlabelsize = fs, ylabelsize = fs, zlabelsize = fs,
    xlabeloffset = 50, ylabeloffset = 50, zlabeloffset = 70,
    )

# Same call as plot_flow3D! in the cell above, hands back the progress Observable
progress = plot_growing_flow!(ax_vid, trajs_vid;
    cmap=cmap, crange=crange, linewidth=lw, subsample=5, ms=ms,
    arrow=true, arrowscale=0.4, endpoint=false)

# The bu = 0 surface
mesh!(ax_vid, isosurface_mesh(ur, ar, br); color=(:steelblue3, 0.4), transparency=true)

vals = range(-br, br; length=200)
lines!(ax_vid, vals, vals, vals ./ vals .* 0.1; color=red, linewidth=10)

zlims!(ax_vid, (0, 1.1*ur))

# Wind to the end, pin the box, then rewind: the limits have to be frozen at the
# *final* extent or the box shrinks around the growing lines during the video.
progress[] = 1.0
freeze_limits!(ax_vid)
progress[] = 0.0

record_growth(fig_vid, progress, "flow_growth.mp4";
    nframes=180, framerate=25,
    ax=ax_vid, turn_speed=0.07)   # revolutions per second

In [ ]:
# Video of the a b c figure above
# `trajs_vid` is the one simulated for the λ a b video above, so the two clips
# are the same flow from the two projections.

# figure_padding = 0 and a square frame keep the white border small to begin
# with — the 3:2 frame was mostly empty on the left and right. record_growth
# cuts off whatever is left once the video is recorded.
fig_vid2 = Figure(size=(100*vs, 100*vs); fontsize = 30, figure_padding = 0)
ax_vid2  = Axis3(fig_vid2[1, 1];
    xlabel=L"\lambda", ylabel=L"\kappa", zlabel=L"\mu",
    viewmode = :fit, aspect = :equal,
    xlabelsize = fs, ylabelsize = fs, zlabelsize = fs,
    xlabeloffset = 50, ylabeloffset = 50, zlabeloffset = 70,
    )

# Equal scaling on all three axes, as in the still figure
xlims!(ax_vid2, (-ar, ar))
ylims!(ax_vid2, (-ar, ar))
zlims!(ax_vid2, (-ar, ar))
ax_vid2.aspect = :data

vals = range(-ar, ar; length=200)
lines!(ax_vid2, vals, vals, vals; color=red, linewidth=10)

progress2 = plot_growing_flow!(ax_vid2, trajs_vid;
    indx=(2, 3, 4), cidx=4,
    cmap=cmap, crange=crange, linewidth=lw, subsample=5, ms=ms,
    arrow=true, arrowscale=0.4, endpoint=false)

progress2[] = 0.0

record_growth(fig_vid2, progress2, "flow_growth_abc.mp4";
    nframes=180, framerate=25,
    ax=ax_vid2, turn_speed=0.07)
